# Sicilian NMT with NLLB-200 — zero-shot + LoRA fine-tune

Evaluate **NLLB-200** zero-shot on our Arba-Sicula test set, then **LoRA fine-tune** and re-evaluate (BLEU/chrF). Sicilian = `scn_Latn`.

Default model here is **`nllb-200-1.3B`** (bigger; fine-tune is tight on a free T4 — if it
OOMs, drop `per_device_train_batch_size` to 2, or use an A100). Switch `MODEL` back to
`nllb-200-distilled-600M` for the lighter run.

**Steps:** Runtime → **GPU**. Run top to bottom; upload the 6 data files via the left Files
panel into `/content` if Drive isn't synced. Artifacts save to Drive (and sync to your PC).

Reference (scn→en): floor 5.27 · Sockeye best 9.79 (tok) · NLLB-600M zero-shot 25.63, fine-tuned 28.93 (raw).

In [1]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
# Colab ships an old torchao that breaks peft's LoRA import; remove it (unused here).
!pip -q uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.6 MB/s eta 0:00:00


In [2]:
# Mount Drive (for data in, artifacts out).
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/SicilianNMT-colab'
except Exception:
    OUT = 'sicilian-nllb-out'
os.makedirs(OUT, exist_ok=True)
print('using', OUT)

Mounted at /content/drive
using /content/drive/MyDrive/SicilianNMT-colab


In [3]:
# Read data from Drive; else from /content (upload via the left Files panel).
def read(p): return open(p, encoding='utf-8').read().splitlines()
DATA = f'{OUT}/data'
need = [f'{s}.{l}' for s in ('train', 'valid', 'test') for l in ('scn', 'en')]
if all(os.path.exists(f'{DATA}/{f}') for f in need):
    base = DATA; print('reading data from Drive:', DATA)
elif all(os.path.exists(f) for f in need):
    base = '.'; print('reading data from /content')
else:
    print('Data not in Drive or /content. Upload the 6 files via the left Files panel, then re-run this cell.')
    from google.colab import files; files.upload(); base = '.'
splits = {s: {l: read(f'{base}/{s}.{l}') for l in ('scn', 'en')} for s in ('train', 'valid', 'test')}
print({s: len(d['scn']) for s, d in splits.items()})

Data not in Drive or /content. Upload the 6 files via the left Files panel, then re-run this cell.


Saving itsc.scn to itsc.scn
Saving test.en to test.en
Saving test.scn to test.scn
Saving test.tsv to test.tsv
Saving train.en to train.en
Saving train.scn to train.scn
Saving train.tsv to train.tsv
Saving valid.en to valid.en
Saving valid.scn to valid.scn
Saving valid.tsv to valid.tsv
{'train': 27386, 'valid': 1000, 'test': 1000}


In [4]:
import json, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import BLEU, CHRF

MODEL = 'facebook/nllb-200-1.3B'   # or 'facebook/nllb-200-distilled-600M'
LANG = {'scn': 'scn_Latn', 'en': 'eng_Latn'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('WARNING: NO GPU. Runtime -> Change runtime type -> GPU, then Restart and run all.')
print('downloading/loading', MODEL, '(several GB the first time)...')
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device).eval()
results = {}

def translate(texts, src, tgt, bs=16, max_len=160):
    tok.src_lang = LANG[src]
    tgt_id = tok.convert_tokens_to_ids(LANG[tgt])
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id, max_length=max_len, num_beams=5)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'  translated {min(i+bs, len(texts))}/{len(texts)}', end='\r')
    print()
    return out

def report(hyp, ref, tag):
    b = BLEU().corpus_score(hyp, [ref]).score
    c = CHRF().corpus_score(hyp, [ref]).score
    results[tag] = {'bleu': round(b, 2), 'chrf': round(c, 2)}
    fname = tag.replace(' ', '_').replace('->', '2')
    open(f'{OUT}/hyp_{fname}.txt', 'w', encoding='utf-8').write('\n'.join(hyp) + '\n')
    json.dump(results, open(f'{OUT}/results.json', 'w'), indent=2, ensure_ascii=False)
    print(f'{tag}:  BLEU={b:.2f}  chrF={c:.2f}   (saved to {OUT})')
print('ready on', device)

downloading/loading facebook/nllb-200-1.3B (several GB the first time)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

ready on cuda


In [5]:
# ---- ZERO-SHOT ----  (a few minutes on GPU; progress prints below)
report(translate(splits['test']['scn'], 'scn', 'en'), splits['test']['en'],  'zero-shot scn->en 1.3B')
report(translate(splits['test']['en'],  'en', 'scn'), splits['test']['scn'], 'zero-shot en->scn 1.3B')

  translated 1000/1000
zero-shot scn->en 1.3B:  BLEU=29.00  chrF=55.23   (saved to /content/drive/MyDrive/SicilianNMT-colab)
  translated 1000/1000
zero-shot en->scn 1.3B:  BLEU=9.89  chrF=41.10   (saved to /content/drive/MyDrive/SicilianNMT-colab)


## LoRA fine-tune (scn→en)

1.3B on a T4 needs a small batch + gradient checkpointing. If it still OOMs, set
`per_device_train_batch_size=2, gradient_accumulation_steps=8`.

In [6]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

torch.cuda.empty_cache()
DIRECTION, EPOCHS = 'scn2en', 3
src, tgt = DIRECTION.split('2')
tok.src_lang, tok.tgt_lang = LANG[src], LANG[tgt]

ft = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                                      target_modules=['q_proj', 'v_proj'], task_type='SEQ_2_SEQ_LM'))
ft.config.use_cache = False
ft.enable_input_require_grads()
ft.print_trainable_parameters()

def tokenize(b):
    return tok(b['src'], text_target=b['tgt'], max_length=128, truncation=True)
train_ds = Dataset.from_dict({'src': splits['train'][src], 'tgt': splits['train'][tgt]}).map(
    tokenize, batched=True, remove_columns=['src', 'tgt'])

args = Seq2SeqTrainingArguments(
    output_dir=f'{OUT}/trainer-{DIRECTION}-1.3B', num_train_epochs=EPOCHS,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=2e-4, fp16=True, logging_steps=50, save_strategy='no', report_to=[])
Seq2SeqTrainer(model=ft, args=args, train_dataset=train_ds,
               data_collator=DataCollatorForSeq2Seq(tok, model=ft)).train()

trainable params: 4,718,592 || all params: 1,375,356,928 || trainable%: 0.3431


Map:   0%|          | 0/27386 [00:00<?, ? examples/s]

Step,Training Loss
50,7.693856
100,7.464248
150,7.147955
200,6.872421
250,6.645918
300,6.955088
350,6.933958
400,6.999553
450,6.695756
500,6.632374


TrainOutput(global_step=5136, training_loss=6.429627552210728, metrics={'train_runtime': 10696.4669, 'train_samples_per_second': 7.681, 'train_steps_per_second': 0.48, 'total_flos': 2.945024619302093e+16, 'train_loss': 6.429627552210728, 'epoch': 3.0})

In [7]:
# ---- EVALUATE FINE-TUNED + SAVE ADAPTER ----
model = ft.eval(); model.config.use_cache = True
report(translate(splits['test'][src], src, tgt), splits['test'][tgt], f'LoRA {DIRECTION} 1.3B')
ft.save_pretrained(f'{OUT}/nllb-lora-{DIRECTION}-1.3B')
print('saved LoRA adapter + results.json + hyps to', OUT)
print(json.dumps(results, indent=2, ensure_ascii=False))

  translated 1000/1000
LoRA scn2en 1.3B:  BLEU=31.16  chrF=56.79   (saved to /content/drive/MyDrive/SicilianNMT-colab)
saved LoRA adapter + results.json + hyps to /content/drive/MyDrive/SicilianNMT-colab
{
  "zero-shot scn->en 1.3B": {
    "bleu": 29.0,
    "chrf": 55.23
  },
  "zero-shot en->scn 1.3B": {
    "bleu": 9.89,
    "chrf": 41.1
  },
  "LoRA scn2en 1.3B": {
    "bleu": 31.16,
    "chrf": 56.79
  }
}


Artifacts under `MyDrive/SicilianNMT-colab/` sync to your PC.

**Note:** NLLB numbers are raw text (compare to the raw floor 5.27); the Sockeye 9.79 was tokenized space.